In [1]:
# notebook: 10_figures_5_2b_fix_5_4_5_5.ipynb
# ============================================================================
# CMVTS Extension — fix Fig 5.2b label overlap + add Fig 5.4 (weight
#                    insensitivity) and Fig 5.5 (vintage stability)
# ----------------------------------------------------------------------------
# Same house style as notebooks 08/09:
#   seaborn greyscale, NO caption in image, dpi=600, png+pdf, legend at BOTTOM.
# ============================================================================

import numpy as np
import pandas as pd
from scipy import stats
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

FIG_DIR = "."
FIG_DPI = 600

sns.set_theme(style="whitegrid")
plt.rcParams.update({
    "font.size":11, "axes.edgecolor":"0.2", "axes.linewidth":0.8,
    "grid.color":"0.85", "figure.dpi":120,
})
def save(fig, name):
    for ext in ("png","pdf"):
        fig.savefig(f"{FIG_DIR}/{name}.{ext}", dpi=FIG_DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"saved {name}.png / {name}.pdf")

# ============================================================================
# FIGURE 5.2b (fixed) — outcome vs card-activity penetration, manual label offsets
# ============================================================================
CARD_ACTIVE = {
    "Indonesia":27.4,"Thailand":46.8,"Viet Nam":58.4,"Philippines":11.8,
    "Bangladesh":9.1,"Cambodia":25.4,"Nepal":36.8,"Pakistan":8.4,"Lao PDR":26.5,
}
Y_DIVERGENCE = {
    "Indonesia":0.0708,"Thailand":0.0092,"Viet Nam":0.0000,"Philippines":0.1809,
    "Bangladesh":0.2110,"Cambodia":0.0810,"Nepal":0.0332,"Pakistan":0.2192,"Lao PDR":0.0753,
}
# manual (dx,dy) label offsets in points, to de-overlap the central & top-left clusters
OFFSETS = {
    "Pakistan":   (6, 6),   "Bangladesh": (8,-10),
    "Philippines":(6, 4),
    "Cambodia":   (-4, 12), "Lao PDR":    (10, 6), "Indonesia":(8,-12),
    "Nepal":      (6, 5),   "Thailand":   (6, 6),  "Viet Nam": (-52, 4),
}
cx = pd.Series(CARD_ACTIVE); cy = pd.Series(Y_DIVERGENCE)
fig, ax = plt.subplots(figsize=(6.6,5.2))
ax.scatter(cx, cy, s=90, c="0.35", edgecolors="black", linewidths=0.8, zorder=3)
b,a = np.polyfit(cx,cy,1); xs=np.linspace(cx.min(),cx.max(),50)
ax.plot(xs,a+b*xs,color="0.1",lw=1.3,zorder=2)
for c in cx.index:
    dx,dy = OFFSETS.get(c,(4,4))
    ax.annotate(c,(cx[c],cy[c]),xytext=(dx,dy),textcoords="offset points",
                fontsize=8.5,color="0.1",
                arrowprops=dict(arrowstyle="-",lw=0.4,color="0.5",
                                shrinkA=0,shrinkB=2) if abs(dx)>20 or abs(dy)>10 else None)
ax.set_xlabel("Target card-activity penetration (%)")
ax.set_ylabel("Realized behavioural divergence  (JSD)")
fig.tight_layout()
save(fig, "fig_5_2b_outcome_vs_penetration")

# ============================================================================
# FIGURE 5.4 — weight insensitivity: |Spearman| & Pearson across macro weights
# ============================================================================
w2 = np.round(np.arange(0,1.0001,0.05),2)
spearman = np.array([-0.883,-0.883,-0.833,-0.833,-0.800,-0.800,-0.817,-0.817,-0.817,
                     -0.817,-0.817,-0.817,-0.817,-0.817,-0.817,-0.817,-0.817,-0.817,
                     -0.817,-0.817,-0.817])
pearson  = np.array([-0.787,-0.786,-0.785,-0.784,-0.783,-0.782,-0.780,-0.779,-0.777,
                     -0.776,-0.775,-0.773,-0.772,-0.771,-0.770,-0.769,-0.768,-0.767,
                     -0.766,-0.765,-0.764])

fig, ax = plt.subplots(figsize=(6.6,4.8))
ax.plot(w2, spearman, color="0.15", lw=1.6, marker="o", ms=5,
        mfc="0.15", mec="black", mew=0.6, label="Spearman $\\rho$", zorder=3)
ax.plot(w2, pearson,  color="0.55", lw=1.6, marker="s", ms=5,
        mfc="0.55", mec="black", mew=0.6, label="Pearson r", zorder=3)
# reference band: all values stay strong & negative
ax.axhspan(-0.90, -0.80, color="0.90", zorder=0)
ax.axhline(-0.80, color="0.4", ls="--", lw=0.9, zorder=1)
# mark equal-weight (paper macro split)
ax.axvline(0.5, color="0.4", ls=":", lw=0.9, zorder=1)
ax.annotate("equal weight\n(primary)", (0.5,-0.86), fontsize=8, ha="center", color="0.2")
ax.set_xlabel("Weight on $C_2$  ($w_2$; $w_3 = 1-w_2$)")
ax.set_ylabel("Correlation with realized divergence")
ax.set_ylim(-0.92, -0.74)
ax.legend(title=None, loc="upper center", bbox_to_anchor=(0.5,-0.13),
          ncol=2, frameon=False, columnspacing=2.0)
fig.tight_layout()
save(fig, "fig_5_4_weight_insensitivity")

# ============================================================================
# FIGURE 5.5 — vintage stability: 2021 vs latest macro-CMVTS per country
# ============================================================================
VINT = {  # (2021, latest)
    "Indonesia":(0.6854,0.6734),"Thailand":(0.7745,0.7762),"Viet Nam":(0.6790,0.6842),
    "Philippines":(0.6008,0.5650),"Bangladesh":(0.5598,0.5561),"Cambodia":(0.5704,0.6034),
    "Nepal":(0.5967,0.5874),"Pakistan":(0.4428,0.4412),"Lao PDR":(0.5528,0.5434),
}
order = sorted(VINT, key=lambda c: VINT[c][1])   # by latest value
y = np.arange(len(order))
fig, ax = plt.subplots(figsize=(6.6,5.0))
for i,c in enumerate(order):
    v21, vl = VINT[c]
    # connector line
    ax.plot([v21,vl],[i,i], color="0.6", lw=1.2, zorder=1)
    ax.scatter(v21, i, s=70, marker="o", c="0.75", edgecolors="black",
               linewidths=0.7, zorder=3, label="2021 vintage" if i==0 else None)
    ax.scatter(vl,  i, s=70, marker="D", c="0.20", edgecolors="black",
               linewidths=0.7, zorder=3, label="latest vintage" if i==0 else None)
ax.set_yticks(y); ax.set_yticklabels(order)
ax.set_xlabel("macro-CMVTS")
ax.set_ylabel("")
# annotate mean shift as an in-axes note (not a caption)
mshift = np.mean([abs(vl-v21) for v21,vl in VINT.values()])
ax.text(0.02, 0.02, f"mean |shift| = {mshift:.3f}", transform=ax.transAxes,
        fontsize=9, va="bottom", ha="left",
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="0.5", lw=0.7))
ax.legend(title=None, loc="upper center", bbox_to_anchor=(0.5,-0.11),
          ncol=2, frameon=False, columnspacing=2.0)
fig.tight_layout()
save(fig, "fig_5_5_vintage_stability")

print("\nAll figures: greyscale, dpi=600, png+pdf, legend bottom, no captions.")

saved fig_5_2b_outcome_vs_penetration.png / fig_5_2b_outcome_vs_penetration.pdf
saved fig_5_4_weight_insensitivity.png / fig_5_4_weight_insensitivity.pdf
saved fig_5_5_vintage_stability.png / fig_5_5_vintage_stability.pdf

All figures: greyscale, dpi=600, png+pdf, legend bottom, no captions.


In [2]:
# --- Fig 5.5 fix: move the stats box off the Pakistan row -------------------
# (re-run only the annotation part; everything else identical to notebook 10)

import numpy as np, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt, seaborn as sns

FIG_DPI = 600
sns.set_theme(style="whitegrid")
plt.rcParams.update({"font.size":11,"axes.edgecolor":"0.2","axes.linewidth":0.8,
                     "grid.color":"0.85","figure.dpi":120})

VINT = {
    "Indonesia":(0.6854,0.6734),"Thailand":(0.7745,0.7762),"Viet Nam":(0.6790,0.6842),
    "Philippines":(0.6008,0.5650),"Bangladesh":(0.5598,0.5561),"Cambodia":(0.5704,0.6034),
    "Nepal":(0.5967,0.5874),"Pakistan":(0.4428,0.4412),"Lao PDR":(0.5528,0.5434),
}
order = sorted(VINT, key=lambda c: VINT[c][1])
y = np.arange(len(order))
fig, ax = plt.subplots(figsize=(6.6,5.0))
for i,c in enumerate(order):
    v21, vl = VINT[c]
    ax.plot([v21,vl],[i,i], color="0.6", lw=1.2, zorder=1)
    ax.scatter(v21, i, s=70, marker="o", c="0.75", edgecolors="black",
               linewidths=0.7, zorder=3, label="2021 vintage" if i==0 else None)
    ax.scatter(vl,  i, s=70, marker="D", c="0.20", edgecolors="black",
               linewidths=0.7, zorder=3, label="latest vintage" if i==0 else None)
ax.set_yticks(y); ax.set_yticklabels(order)
ax.set_xlabel("macro-CMVTS"); ax.set_ylabel("")

# widen x-limits so nothing sits at the very edge, then place box in EMPTY upper-right
ax.set_xlim(0.42, 0.80)
ax.margins(y=0.08)
mshift = np.mean([abs(vl-v21) for v21,vl in VINT.values()])
ax.text(0.97, 0.06, f"mean |shift| = {mshift:.3f}", transform=ax.transAxes,
        fontsize=9, va="bottom", ha="right",         # <-- right-anchored, clear of points
        bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="0.5", lw=0.7))
ax.legend(title=None, loc="upper center", bbox_to_anchor=(0.5,-0.11),
          ncol=2, frameon=False, columnspacing=2.0)
fig.tight_layout()
for ext in ("png","pdf"):
    fig.savefig(f"fig_5_5_vintage_stability.{ext}", dpi=FIG_DPI, bbox_inches="tight")
plt.close(fig)
print("saved fig_5_5_vintage_stability.png / .pdf (box moved to lower-right, clear of Pakistan)")

saved fig_5_5_vintage_stability.png / .pdf (box moved to lower-right, clear of Pakistan)
